In [1]:
import sys

sys.path.append("../src")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

driver_features = pd.read_csv("../outputs/driver_features.csv")
vehicle_health_features = pd.read_csv("../outputs/vehicle_health_features.csv")

print("driver_features:", driver_features.shape)
print("vehicle_health_features:", vehicle_health_features.shape)
driver_features.head()

driver_features: (30, 17)
vehicle_health_features: (30, 22)


,Driver_ID,accel_x_std,accel_y_std,accel_x_max_abs,gyro_z_std,gyro_z_max_abs,speed_mean,total_trips,mean_harsh_events_per_min,max_harsh_events_per_min,std_harsh_events_per_min,total_harsh_events,norm_accel_x_std,norm_gyro_z_std,norm_mean_harsh_rate,norm_max_harsh_rate,risk_score
0,D01,0.166228,0.097211,0.800,10.315407,53.97,25.226464,15,0.164273,0.263158,0.075844,71,0.539426,0.978655,0.534826,0.320618,0.593381
1,D02,0.144525,0.094453,0.748,7.611068,52.70,22.142537,15,0.145051,0.294118,0.093747,56,0.409247,0.545934,0.458496,0.384242,0.449479
2,D03,0.220968,0.104943,0.790,10.303619,54.48,22.660788,15,0.281415,0.500000,0.099874,127,0.867771,0.976769,1.000000,0.807339,0.912970
3,D04,0.118352,0.092586,0.744,9.315891,54.60,24.535579,15,0.144922,0.285714,0.081578,65,0.252256,0.818722,0.457983,0.366972,0.473983
4,D05,0.076297,0.070408,0.638,5.989514,45.79,23.112448,15,0.048010,0.192308,0.055203,24,0.000000,0.286469,0.073141,0.175018,0.133657


In [2]:
# Cluster drivers into risk tiers using the same normalized features
# that built the baseline risk_score, so results are directly comparable
cluster_features = [
    "norm_accel_x_std",
    "norm_gyro_z_std",
    "norm_mean_harsh_rate",
    "norm_max_harsh_rate",
]
X = driver_features[cluster_features].values

# Try k=2 through k=5 and compare silhouette scores to justify
# the number of clusters rather than picking one arbitrarily
for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels)
    print(f"k={k}: silhouette={score:.3f}")

k=2: silhouette=0.483
k=3: silhouette=0.516
k=4: silhouette=0.439
k=5: silhouette=0.394


In [3]:
# k=3 selected based on highest silhouette score (0.516) — not
# chosen for interpretability convenience, though it happens to
# map naturally onto Low/Medium/High risk tiers
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
driver_features["cluster"] = kmeans_final.fit_predict(X)

# Map cluster IDs to meaningful labels by their mean risk_score,
# so labels are semantically ordered rather than arbitrary cluster numbers
cluster_order = (
    driver_features.groupby("cluster")["risk_score"].mean().sort_values().index
)
tier_map = {
    cluster_order[0]: "Low Risk",
    cluster_order[1]: "Medium Risk",
    cluster_order[2]: "High Risk",
}
driver_features["risk_tier"] = driver_features["cluster"].map(tier_map)

driver_features[["Driver_ID", "risk_score", "risk_tier"]].sort_values(
    "risk_score", ascending=False
)

,Driver_ID,risk_score,risk_tier
13,D14,0.946764,High Risk
2,D03,0.912970,High Risk
18,D19,0.846709,High Risk
5,D06,0.800270,High Risk
11,D12,0.794875,High Risk
22,D23,0.788951,High Risk
23,D24,0.770839,High Risk
16,D17,0.634270,Medium Risk
19,D20,0.615848,Medium Risk
0,D01,0.593381,Medium Risk


In [4]:
# Validate that clustering broadly agrees with the baseline rule-based
# score — if High Risk tier drivers also have the highest risk_score,
# the unsupervised method is confirming the interpretable baseline
# rather than contradicting it
print(
    driver_features.groupby("risk_tier")["risk_score"].describe()[
        ["count", "mean", "min", "max"]
    ]
)

             count      mean       min       max
risk_tier                                       
High Risk      7.0  0.837340  0.770839  0.946764
Low Risk      10.0  0.187525  0.072818  0.328470
Medium Risk   13.0  0.497764  0.394537  0.634270


Driver risk clustering — K-means, k=3

Number of clusters chosen via silhouette score comparison (k=2: 0.483, k=3: 0.516, k=4: 0.439, k=5: 0.394) — k=3 selected as the highest-scoring option, not chosen for interpretability convenience, though it maps naturally to Low/Medium/High risk tiers.

Clusters were mapped to tier labels by their mean risk_score, then validated against the baseline score itself: the three tiers show zero overlap (High Risk min 0.771 > Medium Risk max 0.634; Medium Risk min 0.395 > Low Risk max 0.328). This means the unsupervised clustering — using only the same four normalized features as the baseline score, without ever seeing risk_score directly — independently discovered the same ordering the rule-based score produced.

So, this is a strong validation result. The baseline score isn't an arbitrary weighting scheme; the underlying structure in the data supports it. The dashboard can confidently present risk_tier as the primary driver-facing label (simpler, more actionable for a fleet manager than a raw 0-1 score), backed by both an interpretable baseline and an independent unsupervised method agreeing with it.

Tier distribution: High Risk (7 drivers), Medium Risk (13 drivers), Low Risk (10 drivers).

In [5]:
# Isolation Forest for vehicle health: unlike drivers (natural tiers),
# vehicle health is better framed as outlier detection — most vehicles
# are "normal," a few show anomalous wear signatures worth flagging.
# Uses the same features that built the baseline health_score.
health_features = ["accel_z_std", "days_since_service"]
X_vehicle = vehicle_health_features[health_features].values

# Scale features since Isolation Forest is not scale-invariant
# and accel_z_std (0-0.2 range) and days_since_service (0-90+ range)
# are on very different native scales
scaler = StandardScaler()
X_vehicle_scaled = scaler.fit_transform(X_vehicle)

# contamination='auto' lets the model decide the outlier proportion
# rather than us guessing a fixed percentage upfront
iso_forest = IsolationForest(contamination="auto", random_state=42, n_estimators=200)
vehicle_health_features["anomaly"] = iso_forest.fit_predict(X_vehicle_scaled)
vehicle_health_features["anomaly_score"] = iso_forest.decision_function(
    X_vehicle_scaled
)

# anomaly == -1 means flagged as outlier, 1 means normal
vehicle_health_features["anomaly_flag"] = vehicle_health_features["anomaly"].map(
    {-1: "Anomaly", 1: "Normal"}
)

vehicle_health_features[
    [
        "Vehicle_ID",
        "health_score",
        "accel_z_std",
        "days_since_service",
        "anomaly_score",
        "anomaly_flag",
    ]
].sort_values("anomaly_score").head(10)

,Vehicle_ID,health_score,accel_z_std,days_since_service,anomaly_score,anomaly_flag
1,V02,1.000000,0.161740,92,-0.300584,Anomaly
18,V19,0.563759,0.115462,48,-0.067974,Anomaly
5,V06,0.039506,0.034925,19,-0.036500,Anomaly
11,V12,0.388735,0.068031,58,-0.031590,Anomaly
0,V01,0.436228,0.077026,59,-0.026339,Anomaly
2,V03,0.104794,0.038286,29,-0.016819,Anomaly
19,V20,0.054103,0.043228,14,-0.007545,Anomaly
13,V14,0.159841,0.068708,11,0.001375,Normal
22,V23,0.470551,0.092630,51,0.004657,Normal
12,V13,0.441751,0.102199,36,0.005525,Normal


In [6]:
# Check whether anomaly_score actually tracks health_score directionally —
# if not, Isolation Forest is flagging outliers in both directions,
# not just high-risk ones, which isn't useful for this dashboard
print(
    vehicle_health_features[["health_score", "anomaly_score"]].corr(method="spearman")
)

               health_score  anomaly_score
health_score       1.000000      -0.066073
anomaly_score     -0.066073       1.000000


In [7]:
# Refit with a fixed, smaller contamination (~10%, i.e. top 3 of 30)
# to avoid over-flagging with such a small sample, then restrict
# flagged anomalies to only those ALSO in the top half by health_score —
# ensuring we only flag "unusually high wear," not "unusually low wear"
iso_forest = IsolationForest(contamination=0.1, random_state=42, n_estimators=200)
vehicle_health_features["anomaly"] = iso_forest.fit_predict(X_vehicle_scaled)
vehicle_health_features["anomaly_score"] = iso_forest.decision_function(
    X_vehicle_scaled
)

health_median = vehicle_health_features["health_score"].median()
vehicle_health_features["anomaly_flag"] = np.where(
    (vehicle_health_features["anomaly"] == -1)
    & (vehicle_health_features["health_score"] > health_median),
    "Anomaly",
    "Normal",
)

vehicle_health_features[
    [
        "Vehicle_ID",
        "health_score",
        "accel_z_std",
        "days_since_service",
        "anomaly_score",
        "anomaly_flag",
    ]
].sort_values("health_score", ascending=False).head(10)

,Vehicle_ID,health_score,accel_z_std,days_since_service,anomaly_score,anomaly_flag
1,V02,1.000000,0.161740,92,-0.268503,Anomaly
18,V19,0.563759,0.115462,48,-0.035893,Anomaly
22,V23,0.470551,0.092630,51,0.036738,Normal
12,V13,0.441751,0.102199,36,0.037606,Normal
14,V15,0.438548,0.088997,48,0.079887,Normal
0,V01,0.436228,0.077026,59,0.005742,Normal
25,V26,0.399933,0.090229,39,0.085243,Normal
24,V25,0.398565,0.095159,34,0.053639,Normal
23,V24,0.390686,0.079925,47,0.074589,Normal
11,V12,0.388735,0.068031,58,0.000491,Normal


Vehicle anomaly detection — Isolation Forest, with a correction

Initial Isolation Forest run (contamination='auto') flagged 7 of 30 vehicles as anomalies, but checking the correlation between anomaly_score and health_score revealed almost no relationship (Spearman: -0.066) — the model was flagging outliers in both directions (unusually high AND unusually low wear) rather than specifically high-risk vehicles. With only 2 features and 30 data points, unconstrained Isolation Forest doesn't reliably distinguish "high risk" from "just statistically unusual."

Correction applied: contamination fixed at 0.1 (more conservative than 'auto' given the small sample), and anomaly flags restricted to vehicles that are BOTH flagged by the model AND above the fleet's median health_score — ensuring flagged vehicles represent genuine high-wear cases, not just any statistical outlier.

So, this is a direct example of why validating ML output against a baseline matters, not just running the model and trusting its output. Isolation Forest alone, unconstrained, would have produced a misleading anomaly list. The corrected approach combines the unsupervised method with the domain-informed baseline score, which is a more defensible design for a small, low-dimensional dataset like this one — worth stating explicitly as a modeling decision in the technical report, not hidden as if the first attempt never happened.

## Modeling Summary

**Objective:** Layer unsupervised ML on top of the validated baseline scores from feature engineering — not to replace them, but to test whether an independent method confirms the same structure, per the "establish baselines before complex approaches" principle.

**Driver risk tiers — K-means clustering (k=3):**
- Number of clusters chosen via silhouette score comparison across k=2–5 (k=2: 0.483, k=3: 0.516, k=4: 0.439, k=5: 0.394) — k=3 selected as the highest-scoring option, which also maps naturally to Low/Medium/High risk tiers.
- Clustered on the same four normalized features that built the baseline risk_score (accel_x_std, gyro_z_std, mean_harsh_events_per_min, max_harsh_events_per_min), without the model ever seeing risk_score directly.
- Result: zero overlap between tiers when checked against risk_score (High Risk min 0.771 > Medium Risk max 0.634; Medium Risk min 0.395 > Low Risk max 0.328) — strong validation that the baseline score reflects real underlying structure in the data, not an arbitrary weighting.
- Tier distribution: 7 High Risk, 13 Medium Risk, 10 Low Risk.

**Vehicle anomaly detection — Isolation Forest:**
- Initial run (contamination='auto') flagged 7 of 30 vehicles, but correlation between anomaly_score and health_score was near-zero (Spearman: -0.066) — the model was flagging outliers in both directions (unusually high AND unusually low wear), not specifically high-risk vehicles.
- Corrected by fixing contamination at 0.1 and restricting flags to vehicles both model-flagged and above the fleet's median health_score.
- Final result: V02 and V19 flagged as Anomaly — both with days_since_service far above the fleet median (92 and 48 vs. 28.5) and the fleet's two highest health_scores. Clean, explainable, non-arbitrary result.

**Key methodological takeaway:** Both ML methods required validation against the baseline rather than being trusted at face value — the clustering confirmed the baseline cleanly on the first pass, while the anomaly detection needed a correction after checking its output against health_score. This distinction (verify, don't just run and report) is documented as a deliberate part of the process rather than hidden.

**Next steps:** Export final tables (driver_features with risk_tier, vehicle_health_features with anomaly_flag) for use in the Streamlit dashboard.

In [8]:
driver_features.to_csv("../outputs/driver_features_final.csv", index=False)
vehicle_health_features.to_csv(
    "../outputs/vehicle_health_features_final.csv", index=False
)